# Неделя 2 — Сборка обучающей выборки

Карточка недели: `docs/week2_dataset.md` · Полное описание: тетрадь, раздел 6

**Конвенция ячеек:**
- ячейка без пометки — готовый код, просто запустите
- `# TODO:` — здесь пишете вы

## 1. Определение таргета

**Объект наблюдения:** клиент (client_id). Одна строка = клиент × snapshot.

**Событие оттока:** клиент считается ушедшим в окне прогноза, если:
1. У него нет ни одной строки в usage за окно прогноза, ИЛИ
2. Средний revenue за окно прогноза < 20% от среднего за окно наблюдения.

**Окна:** наблюдение 6 месяцев (включая snapshot), зазор 2 месяца, прогноз 3 месяца.

**Почему так:** порог 20% отделяет реальное затухание от сезонных колебаний; зазор в 2 месяца оставляет отделу удержания время на реакцию.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append('..')
from src.features import (FORECAST_MONTHS, GAP_MONTHS, OBSERVATION_MONTHS,
                          SNAPSHOTS, build_snapshot, window_bounds)

clients = pd.read_parquet('../data/processed/clients.parquet')
usage = pd.read_parquet('../data/processed/usage.parquet')
print(clients.shape, usage.shape)

(50400, 8) (1073079, 7)


## 2. Проверка границ окон

Прежде чем собирать выборку, убедитесь, что `window_bounds` считает то, что нужно.

In [2]:
# TODO: реализуйте window_bounds() в src/features.py, затем раскомментируйте
print(window_bounds('2025-01'))
# ожидаем ('2024-08', '2025-01', '2025-04', '2025-06')

('2024-08', '2025-01', '2025-04', '2025-06')


## 3. Один snapshot — глазами

Соберите ОДИН снапшот и посмотрите на него, прежде чем запускать цикл по семи.

In [3]:
# TODO: реализуйте make_features(), make_target(), build_snapshot(), затем:
one = build_snapshot(usage, clients, '2025-01')
print(one.shape)
print('доля оттока:', one['target'].mean().round(4))
one.head(20)

(35788, 22)
доля оттока: 0.0925


,client_id,revenue_mean,revenue_min,revenue_max,revenue_std,revenue_count,n_sim_last,n_sim_mean,debt_mean,debt_last,...,revenue_trend,months_declining,revenue_trend_norm,n_sim_trend,debt_months,target,snapshot_date,segment,product,region
0,CL000001,1.819404e+06,1753016.27,1889117.87,55037.813331,6,74,70.000000,153.695000,0.00,...,-28857.501714,5,-0.015861,5.714286e-02,1,0,2025-01,крупный бизнес,телефония,Юг
1,CL000002,2.009294e+06,1904798.92,2102392.81,85605.497301,6,104,107.500000,0.000000,0.00,...,14274.025429,2,0.007104,-3.000000e+00,0,0,2025-01,крупный бизнес,телефония,Юг
2,CL000004,1.096042e+05,104496.60,116423.51,4144.058844,6,17,16.500000,0.000000,0.00,...,-1719.306571,1,-0.015687,1.428571e-01,0,0,2025-01,средний бизнес,облачные сервисы,Сибирь
3,CL000006,1.358691e+05,121663.35,150270.21,10782.764861,6,23,23.833333,246.181667,0.00,...,-5723.291714,5,-0.042124,2.857143e-02,1,0,2025-01,средний бизнес,телефония,Москва
4,CL000007,2.295501e+05,199371.75,248463.70,16542.550028,6,10,9.833333,0.000000,0.00,...,7457.920286,1,0.032489,-8.571429e-02,0,0,2025-01,средний бизнес,интернет,Юг
5,CL000008,4.303366e+05,373252.06,458550.61,30856.417549,6,45,49.166667,119.568333,0.00,...,-8019.146857,0,-0.018635,-4.285714e-01,1,0,2025-01,крупный бизнес,облачные сервисы,Урал
6,CL000010,5.351594e+04,2079.37,98022.72,38870.871910,6,8,13.500000,701.835000,1687.84,...,-20574.908000,5,-0.384463,-2.714286e+00,3,1,2025-01,средний бизнес,мобильная связь,Москва
7,CL000011,3.953851e+05,272123.22,576622.76,143600.659339,4,234,248.750000,0.000000,0.00,...,80471.764000,1,0.203528,-6.900000e+00,0,1,2025-01,крупный бизнес,облачные сервисы,Сибирь
8,CL000012,6.213862e+04,58545.76,65731.49,5081.078411,2,10,9.500000,0.000000,0.00,...,-7185.730000,1,-0.115640,1.000000e+00,0,0,2025-01,средний бизнес,телефония,Юг
9,CL000013,1.061114e+05,99106.30,112517.06,4476.658698,6,23,22.333333,0.000000,0.00,...,1791.458857,0,0.016883,2.857143e-01,0,0,2025-01,средний бизнес,облачные сервисы,Сибирь


## 4. Сборка по всем снапшотам

In [4]:
# Собираем по всем снапшотам
dataset = pd.concat([build_snapshot(usage, clients, s) for s in SNAPSHOTS],
                    ignore_index=True)
dataset.to_parquet('../data/processed/dataset.parquet')
print(f"Размер датасета: {dataset.shape}")

Размер датасета: (257631, 22)


## 5. Обязательные проверки

Ожидаемая картина: доля оттока по снапшотам меняется плавно.
Медленный дрейф — норма. Скачки в разы — баг.

In [5]:
print(dataset.groupby('snapshot_date')['target'].agg(['mean', 'count']).round(4))

                 mean  count
snapshot_date               
2024-07        0.0721  32156
2024-10        0.0800  34097
2025-01        0.0925  35788
2025-04        0.1056  37297
2025-07        0.1134  38564
2025-10        0.1246  39464
2026-01        0.1379  40265


In [6]:
assert dataset['segment'].isna().sum() == 0, 'потеряли сегмент при merge'
assert not dataset.duplicated(subset=['client_id', 'snapshot_date']).any(), 'дубли клиент x снапшот'
print('проверки пройдены')

проверки пройдены


## 6. Самопроверка на утечку

TODO: таблица по каждому признаку — из каких месяцев считается и мог ли я знать
это значение в день snapshot. Признак с ответом «нет» или «не уверен» в модель не идёт.

| Признак | Из каких месяцев | Мог ли знать в день snapshot? |
|rev_mean|2024-08...2025-01|Да|
|rev_last|2025-01|Да|
|rev_last_to_mean|2024-08 ... 2025-01|Да|
|n_months|2024-08 ... 2025-01|Да|
|segment|Статический|Да|
|product|Статический|Да|
|region|Статический|Да|